Setup and Environment

In [2]:
# Setup and Imports
import sys
import os

current_dir = os.getcwd()

if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root set to: {project_root}")


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

import random
import numpy as np

from src.models import build_normalized_resnet20
from src.defenses.adversarial_training import train_adversarial_epoch, evaluate_robustness

os.makedirs(os.path.join(project_root, "checkpoints"), exist_ok=True)
print("Setup completed successfully!")

Project root set to: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense
Setup completed successfully!


Seed Initialization

In [3]:
# Cell 2: Seed Initialization (Contract Section 23)
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # For maximum reproducibility (Contract Section 23)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


Data Loading

In [5]:
# Cell 3: CIFAR-10 Data Loading (Contract Section 1)
import os

# ToTensor() converts PIL images in [0, 255] to PyTorch Tensors in [0.0, 1.0].
transform = transforms.ToTensor()

# استفاده از project_root برای اشاره دقیق به پوشه data در ریشه پروژه
data_path = os.path.join(project_root, 'data')

full_train_dataset = torchvision.datasets.CIFAR10(
    root=data_path, 
    train=True, 
    download=True, 
    transform=transform
)

# Split into train and validation sets (e.g., 45k train, 5k val)
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
print(f"Data directory points to: {data_path}")

Train samples: 45000, Validation samples: 5000
Data directory points to: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/data


Checkpoint Saving Utility

In [10]:
# Cell 4: Checkpoint Utility (Contract Sections 14 & 15)
import os
import torch
import torch.nn as nn
import torch.optim as optim

def save_checkpoint(
    model: nn.Module, 
    optimizer: optim.Optimizer, 
    epoch: int, 
    best_metric: float, 
    defense_id: str, 
    checkpoint_name: str, 
    config_dict: dict
):
    """
    Saves the checkpoint exactly according to Contract Section 15.
    """
    checkpoint = {
        "model_id": "resnet20",
        "defense_id": defense_id,
        "epoch": epoch,
        # Save only the backbone state_dict as per contract
        "model_state_dict": model.backbone.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
        "config": config_dict,
    }
    
    # Use absolute path based on project_root to prevent directory missing errors
    save_dir = os.path.join(project_root, "checkpoints")
    
    # Ensure the directory exists just in case
    os.makedirs(save_dir, exist_ok=True)
    
    save_path = os.path.join(save_dir, checkpoint_name)
    torch.save(checkpoint, save_path)
    
    print(f"Checkpoint saved: {save_path} (Best Metric: {best_metric:.4f})")

Training the Clean Model (Base Model)

In [11]:
# Cell 5: Train Clean Model (resnet20)
def train_clean_model(epochs=30):
    print("--- Starting Clean Model Training ---")
    model_clean = build_normalized_resnet20(num_classes=10, eval_mode=False).to(device)
    optimizer = optim.SGD(model_clean.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    best_clean_acc = 0.0
    config_dict = {"seed": SEED, "learning_rate": 0.1, "epochs": epochs}

    for epoch in range(1, epochs + 1):
        model_clean.train()
        total_loss, correct, total = 0.0, 0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model_clean(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * labels.size(0)
            correct += logits.argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)
            
        scheduler.step()
        
        # Validation
        model_clean.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits = model_clean(images)
                val_correct += logits.argmax(dim=1).eq(labels).sum().item()
                val_total += labels.size(0)
                
        val_acc = val_correct / val_total
        print(f"Epoch {epoch:02d} | Train Loss: {total_loss/total:.4f} | Val Acc: {val_acc:.4f}")
        
        # Save best checkpoint (Contract Section 14)
        if val_acc > best_clean_acc:
            best_clean_acc = val_acc
            save_checkpoint(
                model_clean, optimizer, epoch, best_clean_acc, 
                "none", "resnet20_clean_best.pt", config_dict
            )

# Uncomment the line below to run clean training
train_clean_model(epochs=30)

--- Starting Clean Model Training ---
Epoch 01 | Train Loss: 1.5547 | Val Acc: 0.5430
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_clean_best.pt (Best Metric: 0.5430)
Epoch 02 | Train Loss: 1.0162 | Val Acc: 0.6138
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_clean_best.pt (Best Metric: 0.6138)
Epoch 03 | Train Loss: 0.7884 | Val Acc: 0.7232
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_clean_best.pt (Best Metric: 0.7232)
Epoch 04 | Train Loss: 0.6675 | Val Acc: 0.7282
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_clean_best.pt (Best Metric: 0.7282)
Epoch 05 | Train Loss: 0.5747 | Val Acc: 0.7454
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_clean_best.pt (Best Metric: 0.7454)
Epoch 06 | Train 

Adversarial Training (PGD-AT)

In [12]:
# Cell 6: Adversarial Training (PGD-AT)
def run_adversarial_training(epochs=30):
    print("--- Starting Adversarial Training (PGD-AT) ---")
    model_at = build_normalized_resnet20(num_classes=10, eval_mode=False).to(device)
    optimizer = optim.SGD(model_at.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    # Contract Section 6 & 15: PGD-AT configurations
    epsilon = 8 / 255
    alpha = 2 / 255
    train_attack_steps = 10
    
    config_dict = {
        "epsilon": epsilon,
        "alpha": alpha,
        "attack_steps": train_attack_steps,
        "seed": SEED,
        "epochs": epochs
    }
    
    best_robust_acc = 0.0

    for epoch in range(1, epochs + 1):
        # 1. Train using adversarial examples
        train_loss, train_robust_acc = train_adversarial_epoch(
            model=model_at,
            dataloader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            epsilon=epsilon,
            alpha=alpha,
            attack_steps=train_attack_steps,
            random_start=True
        )
        
        scheduler.step()
        
        # 2. Evaluate robust accuracy to check for Robust Overfitting
        val_loss, val_robust_acc = evaluate_robustness(
            model=model_at,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
            epsilon=epsilon,
            alpha=alpha,
            attack_steps=20  # Stronger attack for evaluation
        )
        
        print(f"Epoch {epoch:02d} | Train Robust Acc: {train_robust_acc:.4f} | Val Robust Acc: {val_robust_acc:.4f}")
        
        # 3. Save the best model based on validation robust accuracy
        if val_robust_acc > best_robust_acc:
            best_robust_acc = val_robust_acc
            save_checkpoint(
                model_at, optimizer, epoch, best_robust_acc, 
                "pgd_at", "resnet20_pgd_at_eps8_best.pt", config_dict
            )
            
        # 4. Save the last epoch (Contract Section 14)
        if epoch == epochs:
            save_checkpoint(
                model_at, optimizer, epoch, val_robust_acc, 
                "pgd_at", "resnet20_pgd_at_eps8_last.pt", config_dict
            )

# Uncomment the line below to run adversarial training
run_adversarial_training(epochs=30)

--- Starting Adversarial Training (PGD-AT) ---
Epoch 01 | Train Robust Acc: 0.2129 | Val Robust Acc: 0.2154
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_pgd_at_eps8_best.pt (Best Metric: 0.2154)
Epoch 02 | Train Robust Acc: 0.2695 | Val Robust Acc: 0.2812
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_pgd_at_eps8_best.pt (Best Metric: 0.2812)
Epoch 03 | Train Robust Acc: 0.2946 | Val Robust Acc: 0.2694
Epoch 04 | Train Robust Acc: 0.3095 | Val Robust Acc: 0.2946
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_pgd_at_eps8_best.pt (Best Metric: 0.2946)
Epoch 05 | Train Robust Acc: 0.3242 | Val Robust Acc: 0.3096
Checkpoint saved: /mnt/d/university/Term-4/AI/AI-Project-Adversarial-Attack-Defense/checkpoints/resnet20_pgd_at_eps8_best.pt (Best Metric: 0.3096)
Epoch 06 | Train Robust Acc: 0.3378 | Val Robust Acc: 0.3240